In [ ]:
pip install bitsandbytes accelerate transformers scikit-learn statsmodels requests numpy matplotlib

In [ ]:
import torch, csv, gc, json, requests, datetime, re, psutil, subprocess
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers import pipeline
from datetime import datetime
from time import perf_counter

In [ ]:
# Clean up GPU before running model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Load Qwen's Qwen3.1 with 4 billion parameters

# Configure 4-bit quantization
quant_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model_name = 'Qwen/Qwen3-4B'
# INSERT HUGGING FACE TOKEN HERE
token = ""

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=token,
    quantization_config=quant_config,
    dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)

In [ ]:
# Cuda device information commands sourced from here: https://massedcompute.com/faq-answers/?question=How%20to%20check%20CUDA%20device%20information%20in%20PyTorch?
# Utilize torch's cuda to ensure that if installed, the GPU will handle running the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

if device.type == "cuda":
    curr_id = torch.cuda.current_device()
    gpu_name = torch.cuda.get_device_name(curr_id)
else:
    curr_id = -1
    gpu_name = "CPU"

print(f"Device: {device}")
print(f"Device ID: {curr_id}")
print(f"GPU Name: {gpu_name}")



In [ ]:
# This function gets the comments timestamp to sort the maintainer comments
def get_timestamp(comment):
    timestamp = comment.get('created_at') or comment.get('submitted_at')
    if not timestamp:
        return None
    return datetime.fromisoformat(timestamp.replace('Z', '+00:00'))

In [ ]:
def get_diffs(PRNum, set_commits):
    # INSERT GITHUB PAT BELOW
    token = ""
    
    # Set headers to get PR metadata and diff info
    headers = {
        'Authorization': f'token {token}',
        'Accept': 'application/vnd.github.v3.+json'
    }

    comments = []
    
    # Get the PR metadata, base sha, and PR author from the PR metadata using the REST API call found here: https://docs.github.com/en/rest/pulls/pulls?apiVersion=2026-03-10#get-a-pull-request 
    url = f"https://api.github.com/repos/talonhub/community/pulls/{PRNum}"
    response = requests.get(url, headers=headers)
    pr_metadata = response.json()
    base_sha = pr_metadata.get('base', {}).get('sha')
    author = pr_metadata.get('user', {}).get('login')
    pr_description = pr_metadata.get('body', '')
    pr_title = pr_metadata.get('title', '')
    if pr_description:
        pr_description = pr_description.strip()

    # Default to very first commit for target sha
    target_sha = None
    
    # Handle the manually selected diffs for specific PRs
    if str(PRNum) in set_commits:
        target_sha = set_commits[str(PRNum)]
    
    # Handle sorting the comments for the correct commit to compare
    else:
        # Get the issue comments for the PR using the REST API call found here: https://docs.github.com/en/rest/issues/comments?apiVersion=2026-03-10#list-issue-comments
        issue_url = f"https://api.github.com/repos/talonhub/community/issues/{PRNum}/comments"
        issue_data = requests.get(issue_url, headers=headers).json()
    
        #Get the review comments for the PR using the REST API call found here: https://docs.github.com/en/rest/pulls/comments?apiVersion=2026-03-10#list-review-comments-on-a-pull-request
        review_url = f"https://api.github.com/repos/talonhub/community/pulls/{PRNum}/comments"
        review_data = requests.get(review_url, headers=headers).json()
        
        # Get the review summary for the PR using the REST API call found here: https://docs.github.com/en/rest/pulls/reviews?apiVersion=2026-03-10#list-reviews-for-a-pull-request
        summary_url = f"https://api.github.com/repos/talonhub/community/pulls/{PRNum}/reviews"
        summary_data = requests.get(summary_url, headers=headers).json()
    
        comments = issue_data + review_data + summary_data
    
        maintainer_comments = []    
        review_time = None
        # Set the roles that a maintainer could be 
        roles = ['COLLABORATOR', 'MEMBER']
        
        for comment in comments:
            # Get review user
            commenter = comment.get('user', {}).get('login')
            
            # Check comment author_association and ensure author is not commenter 
            if comment.get('author_association') in roles and commenter != author:
                # Add the comment to the comment list to be sorted
                maintainer_comments.append(comment)
    
        #Sort the maintainer comments on timestamp, submitted_at for review summary and created_at for review and issue comments
        maintainer_comments.sort(key=get_timestamp)
    
        #Get the first maintainer comment
        first_maintainer_comment = maintainer_comments[0] 
        
        # Get all commits using the using the REST API call found here: https://docs.github.com/en/rest/pulls/pulls?apiVersion=2026-03-10#list-commits-on-a-pull-request
        commits_url = f"https://api.github.com/repos/talonhub/community/pulls/{PRNum}/commits"
        commits = requests.get(commits_url, headers=headers).json()
        
        
        diffs_patches = "[]"
    
        if not commits:
            return "[]", pr_description
        
        # Get the oldest commit before the first maintainer review 
        elif first_maintainer_comment:
            target_sha = commits[0]['sha']
            # Loop through the commits backwards to latest commit NOT by maintianer before review
            for commit in reversed(commits):
                # Get the time of the commit
                commit_time = datetime.fromisoformat(commit.get('commit', {}).get('committer', {}).get('date').replace('Z', '+00:00'))

                # Get the commit author's association to check against the maintainer roles
                commit_author = commit.get('author_association')
                
                # Check if it is less than the time of the first maintainer review and not a commit made by a maintainer
                if commit_time < get_timestamp(first_maintainer_comment) and commit_author not in roles:
                    target_sha = commit['sha']
                    break
        # Get latest commit if no maintainer review
        else:
            target_sha = commits[-1]['sha']
    
    # Handle no base sha or target sha
    if not base_sha or not target_sha:
        print(f"Skipping PR {PRNum}: Missing SHAs")
        return "[]", pr_description
    
    # Compare the base sha and target sha for diffs using the REST API call found here: https://docs.github.com/en/rest/commits/commits?apiVersion=2026-03-10#compare-two-commits
    url = f"https://api.github.com/repos/talonhub/community/compare/{base_sha}...{target_sha}"
    
    response = requests.get(url, headers=headers)
    
    patches = []
    if response.status_code == 200:
        data = response.json()
        # Get the files from the compare to grab the diff patches per file
        files = data.get('files', [])
        for f in files:
            patch = f.get('patch')
            filename = f.get('filename')
            if patch:
                patches.append(f" FILE: {filename}\n{patch}")
                
        diffs_patches = json.dumps(patches, indent=2)
        
        print("Got diff patches")
    else:
        print(f"Failed to get diff: {response.status_code}")
    return diffs_patches, pr_description, pr_title

In [ ]:
category_prompts = ["Read the PR diffs, description, and title and answer the question with only YES or NO. Also use the following category background information to assist with your applicability decision. Here is some background information on the Voice Coding for the Talonhub/community OSS. Talon community offers commands for inserting code in numerous languages. Formatters make dictating identifiers easier. Formatter commands are useful for formatting text, including prose and identifiers in code. You use a formatter by saying the name of the formatter and then the words you want formatted. Operator commands support dictating things like variable assignment, mathematical operators, and comparison operators. With supported languages, most operators can be dictated with op (operator name) and comparison operators can be dictated with is (operator name). For example, saying op equals while editing a Python file inserts =. Saying is equal inserts ==. To see which operators are supported for a given language and their spoken forms, use the help operators command. To change how operators are inserted, find the Python file for the language implementation in the lang directory and edit the Operators object. Symbol commands allow greater flexibility and handling paired delimiters like parentheses. Symbol commands can be useful for situations not handled by operators and for greater flexibility. help symbols will show you the symbol commands. You can say pad <user.symbol_key> to insert a symbol surrounded by spaces, such as pad star inserting  * . The pad command can also be used by itself to surround the cursor with spaces. Because inserting a pair of symbols around the cursor (such as braces, parentheses, and quotation marks) is so common, community offers support for inserting paired delimiters. You can edit these commands by changing the user.delimiter_pair list. Snippets allow efficiently dictating formulaic code patterns and are used for things like control flow statements, class and function definitions, return statements and more. Snippets insert text with placeholders. Jump to the next placeholder using the snip next command. Insert a snippet by saying snip {user.snippet}, where {user.snippet} is a snippet name. Snippets can be inserted directly through the VSCode snippet system if you have the command server VSCode extension installed. This also lets snip next use VSCode's support for getting to the next snippet placeholder. Without editor support, snip next assumes that the cursor is on the same line as the current snippet placeholder. If you move the cursor to another line first and then use snip next, the cursor may be moved to the wrong position. Community will try to insert snippets manually without editor support. Depending on your editor's settings, inserting a snippet may insert extra delimiters and disrupt indentation. You can set the user.snippet_raw_text_paste setting to true to always insert snippets (in contexts without editor support) through pasting, which will get correct formatting in some editors. You can also change the user.snippet_raw_text_spaces_per_tab setting to determine how many spaces to use per tab; in most cases, you should set this to -1 to use tabs instead of spaces. Most code editors will automatically expand tabs to spaces when pasting code, if so configured. Note that you should probably adjust these settings on a per-editor basis in a separate .talon file for each editor. You might be able to further address some formatting issues from your editor not supporting snippets by using an automatic code formatting extension. The community snippet actions can be useful for custom commands. user.insert_snippet takes the body of a snippet as its argument and inserts the snippet. The following example uses this action to insert a C++ static cast operation. This does not require defining a snippet inside a snippet file. user.insert_snippet_by_name takes the name of a snippet as its first argument and a dictionary of substitutions as an optional second argument. This can be used to insert snippets defined in .snippet files using the names from the name: field. The substitution dictionary for the second argument can be used to replace snippet stops programmatically before inserting a snippet. The following example shows replacing the $1 and $0 snippet stops programmatically. The dictionary maps the stop names to their string replacements. You omit the $ in the stop names. The user.insert_snippet_by_name_with_stop_at_end action is the same as the user.insert_snippet_by_name action but adds a final stop ($0) at the end of the snippet before insertion unless the snippet already ends with the last stop ($0 or highest-numbered). This is the default action used by community for inserting snippets by name with the snip {user.snippet} command. The user.insert_snippet_by_name_with_phrase action takes 2 arguments: the name of the snippet to insert and then a phrase. This inserts the snippet and then replaces the snippet stop that has an associated formatter with the result of applying the formatter to the phrase. For instance, when used with the functionDeclaration snippet, the phrase formatted with the appropriate formatter gets used as the function name. The user.insert_snippet_by_name_with_phrase_and_stop_at_end action does the same thing as user.insert_snippet_by_name_with_phrase but adds a stop at the end of the snippet before insertion unless the snippet already ends with a stop. This is the default action used by community used when inserting snippets that can take a phrase argument, such as snip funk using the dictated phrase as the function name, i.e. snip funk function name. The user.move_cursor_to_next_snippet_stop action moves the cursor to the next snippet stop. This category focuses on identifying when the diff is attempting to provide support for a new language and providing the feedback that the code should build off the existing Talon and Python programming language tags. This category is \”Build off of the existing programming language support when implementing support for a new language\”. Answer YES if the provided diffs, description, and/or title are not utilizing the provided Voice Coding background information when a new language is being implemented. Otherwise, answer NO. The YES/NO answer must be at the beginning of your response.",
                    "Read the PR diffs, description, and title and answer the question with only YES or NO. Also use the following category background information to assist with your applicability decision. Here is some background about useless code. This category focuses on identifying and eliminating code that has no functional purpose or is no longer relevant to the system. This includes unused variables, unused code, redundant definitions, commented out code where both .talon and .py use the hash symbol for comments, unnecessary tags which can include tags that are defined in a .py file but never selected within a .talon file or tags that could have been handled using contexts, or irrelevant overrides which are overrides of an action that have no usable definition. Answer YES if the provided diffs, description, and/or title contains any of the previously defined useless code. Otherwise, answer NO. The YES/NO answer must be at the beginning of your response.",
                    "Read the PR diffs, description, and title and answer the question with only YES or NO. Also use the following category background information to assist with your applicability decision. Look at the PR diff given, analyze if the diff is changing the existing codes functionality and how it would behave without clear explanations in comments in the code or the diff description or without an update to what changes are breaking in comments in the code or a deprecation warning exists in the code as a message to the user. If your analysis indicates the diff code is changing the existing functionality and one or multiple of the lack of proper communication about it is happening answer YES, otherwise answer NO. The YES/NO answer must be at the beginning of your response.",
                    "Read the PR diffs, description, and title and answer the question with only YES or NO. Also use the following category background information to assist with your applicability decision. Consider the Talon Voice Command and Programming Principles when making this analysis: P01 - Prefer [object][verb] rather than [verb][object] for new commands. For example 'file save' is better than 'save file'. It may not sound as natural, but it helps for grouping related commands in lists and avoiding conflicting names. P02 - Use browser.host matcher for web apps. Though this matcher requires a browser extension on some operating systems it is the only unambiguous way of referring to a web app. P03 - Use the app.bundle matcher for apps on OSX. This is the least ambiguous way of referring to a particular program. P04 - Use both app.name and app.exe matchers for apps on Windows. That is the context should OR together one matcher of each type. Apparently the MUICache can break, perhaps making one of these matchers stop working. P05 - Use .talon-list files for Talon lists unless there is a good reason not to. .talon-list files are easier for users to edit — especially users who are not programmers. If a list needs to be constructed or referenced through Python code, it may make sense to instead leave it in Python. P06 - Communicate with maintainers before submitting a large pull request. Not everything that is useful for a specific user belongs in community itself. To avoid wasting your time, it is recommended to file an issue to give a chance for maintainers to respond before doing a lot of work on a feature that may or may not get accepted. Maintainers may additionally have useful suggestions on how to implement your desired changes that can save you considerable work and reduce the amount of changes requested in peer review. P07 - Try to create pull requests that focus on a single feature at a time. Separating pieces of code that do not depend on each other into separate pull requests makes peer review easier and will usually lead to changes getting merged faster. Large or unfocused pull requests trying to do too much at once often lose momentum and become abandoned in peer review. The reason for this is usually that there are too many pieces that need to be properly reviewed before the entire thing can be merged. Smaller, more focused pull requests allow uncontroversial changes to merged quickly while facilitating more focused discussions for those changes that do merit it. P08 - Use the stored state management actions defined in storage_state.py to store state on disk. This keeps stored state within the stored_state directory, which makes it easier for users to keep track of and git ignore. P09 - If there is no reasonable implementation for an action in a context, it should raise an exception. This has the advantage of stopping execution, preventing calling code from continuing with incorrect assumptions. If there is an (inappropriate) implementation for the action active in the context, implement the action in the specific context to raise an exception. You may additionally display a notification if it is helpful to let the user know that the action did not complete successfully. If there is no implementation for the action active in the context, don't implement it. Talon will appropriately raise an exception when executing the action in the context. This diff analysis revolves around the need to avoid unnecessary complexity. Answer YES if the provided diffs, description, or title introduces or could benefit from reducing complexity, avoiding overengineering, simplifying abstractions, or replacing complex logic with simpler alternatives. Otherwise, answer NO. The YES/NO answer must be at the beginning of your response.",
                    "Read the PR diffs, description, and title and answer the question with only YES or NO. Also use the following category background information to assist with your applicability decision. Here is some background information on what an unclear decision in the diff would be. An unclear decision is a code addition, deletion, or update that has any of the following: not been explained in the PR description, missing proper commenting or justification when involving complex logic or a configuration shift, does not align with the goal of the PR description. This category focuses on additional context being required for code additions/deletions/updates that are unnecessary or lack proper justification. This category is \”Document unclear decisions\”. Answer YES if the provided diffs, description, and/or title have any previously described unclear decisions. Otherwise, answer NO. The YES/NO answer must be at the beginning of your response.",
                    "Read the PR diffs, description, and title and answer the question with only YES or NO. Also use the following category background information to assist with your applicability decision. Here is some background about generality or platform support. This category focuses on whether the changes are too narrowly tailored to a specific environment, platform, or use case, and whether they could be made more broadly applicable. Consider the following requirements for this prompt: Is the code, description, or title limited to one OS or app for universal behavior, does the diff, description, or title implement code for one OS like Windows but does not provide an implementation for another OS like MacOS, and check if the context in the .talon or .py files are limiting commands to one app unnecessarily. Some examples of where different behaviors exist on different systems are: Implementing keyboard shortcuts for an application often requires different behavior on MacOS  and Mouse scrolling can behave differently across OSs. This category is \”Improve generality or platform support\”. Answer YES if the provided diffs, description, and/or title match any of the previously defined generality cases. Otherwise, answer NO. The YES/NO answer must be at the beginning of your response.", 
                    "Read the PR diffs, description, and title and answer the question with only YES or NO. Also use the following category background information to assist with your applicability decision. Look at the PR diff given, familiarize yourself with the .py files to see what Action classes are existing and if they are being changed, analyze if the diff is attempting to use if statements to check for certain scenarios in the code that should be using the context to handle those situations. For example if the action code has an if statement to check if the OS is Windows and the application is something like Mozilla Firefox instead of using the built in context feature to say os: windows: application: mozilla firefox: in the header, especially if that scenario handling code in the action code block overrides the existing context and does not add new contexts to override the actions that fit for that context. If your analysis indicates the diff code is trying to override actions inside the action code by checking for scenarios or context in the action code itself and does not use the context feature answer YES, otherwise answer NO. The YES/NO answer must be at the beginning of your response.",
                    "Read the PR diffs, description, and title and answer the question with only YES or NO. Also use the following category background information to assist with your applicability decision. Consider the following pages from Talon documentation when making this analysis. From the page of Tags: Besides concrete features like an application's name or a window's title, a context can also select for tags. Tags have a couple of main uses: Tags can be used to activate additional voice commands within a particular context. For example Talon Community has some tab management commands (e.g. tab new) that apply to many applications. Application specific contexts or .talon files can simply enable the tag (and potentially implement the relevant actions) to activate those voice commands. Tags can be enabled from Python to activate a set of voice commands given certain conditions. For example the mouse grid activates a tag when it is visible. This tag enables the 'grid off' and 'grid reset' commands. To make a tag available, it must first be declared in a module: generic_application_features.py: from talon import Module mod = Module() this declares a tag in the user namespace (i.e. 'user.tabs') mod.tag(\"tabs\", desc=\"basic commands for working with tabs within a window are available\") Next let's define a set of generic voice commands we think will apply to all applications with tabs: tabs.talon: This selects for the tag 'user.tabs'. tag: user.tabs (open | new) tab: app.tab_open() last tab: app.tab_previous() next tab: app.tab_next() close tab: app.tab_close() reopen tab: app.tab_reopen() Finally, let's activate these voice commands for the firefox application: firefox.talon: app: Firefox This activates the tag 'user.tabs'. tag(): user.tabs Of course, the commands we defined in tabs.talon just invoke corresponding actions, so unless the default behavior of those actions is what we want, we'd also need to implement them in a Python file (see Actions). Happily, in this case the default behavior suffices. Tags and actions often go together in this way. There's also the option of enabling tags from within Python. To do that you can use a Context instance like this: from talon import Context ctx = Context() ctx.matches = \"app: Firefox\" You can alter the set of tags whenever you like within your Python code. The tags will only be applied if your Context is currently active and they are included in the tags property. Note that you must replace the entire set of tags at once, you can't individually add and delete them ctx.tags = [\"user.tabs\"] Tags are a commonly used part of the Talon framework. Related but less commonly used are modes and scopes. From the page of Abstractions: In order to script Talon it is useful to understand the abstractions it uses. Let's start by giving you a brief overview of how they fit together. The first concept is the Module. This is used to group behavior like settings, actions, or tags. The Context is a central feature of the Talon framework. A context is the circumstances under which a set of behaviour applies. For example we might only activate some voice commands when the title of the currently focussed window matches a given pattern. The concepts of Tags and Apps, and less commonly Modes and Scopes are all ways of providing information to match against in a Context. The next key component is the implementation of behaviour via Actions. Two examples are moving the mouse cursor and pasting the contents of the clipboard. Talon comes with some built in actions, but most are defined and implemented in user scripts. One of the primary modes of input to Talon is through voice commands defined in .talon files. To implement commands containing dynamic 'variables' (e.g. 'allcaps some arbitrary words') you can utilize Lists and captures. In addition to the above we also have the concept of Settings. Built-in and custom settings are used by actions to configure their behaviour (e.g. to change the delay between key presses in the insert() action). Also on Tags: Many terminal applications are supported out of the box, but you may not want all the commands enabled. To use command sets in your terminal applications, enable/disable the corresponding tags in the terminal application-specific .talon file. tag(): user.file_manager tag(): user.git tag(): user.kubectl tag(): user.tabs For instance, kubectl commands (kubernetes) aren't relevant to everyone. Note also that while some of the command sets associated with these tags are defined in talon files within tags, others, like git, are defined within apps. Commands for tabs are defined in tabs.talon. Unix utilities If you have a Unix (e.g. macOS) or Linux computer, you can enable support for a number of common terminal utilities like cat, tail, or grep by uncommenting the following line in unix_shell.py: # ctx.tags = [\"user.unix_utilities\"]. Once you have uncommented the line, you can customize your utility commands by editing tags/terminal/unix_utility.talon-list. For generalization, functionality is used in multiple contexts. Commands should be put in a place where they can be overwritten from if you activate the tag in a context the command is available this overrides the actions in that context tags are a directory for where the context is. Are commands being directly defined or is it being used as a placeholder where python files can come in and actually specify what each action does with the command based on the device or operating system. This diff analysis revolves around understanding whether there is a need to generalize functionality across applications: Answer YES if the provided diffs, description, and/or title is overly application specific and could be made more reusable, such as by using tags or shared abstractions. Otherwise, answer NO. The YES/NO answer must be at the beginning of your response.",
                    "Read the PR diffs, description, and title and answer the question with only YES or NO. Also use the following category background information to assist with your applicability decision. Here is some background information on what complex code in the diff would be. Complex code is code that demonstrates any of the following: three or greater levels of nested loops/conditional statements, high number of decision points like if/switch/case in a single function, lack of simple and readable logic. Complex code will also lack comments. This category is \”Document complex code\”. Answer YES if the provided diffs, description, and/or title have any previously described complex code. Otherwise, answer NO. The YES/NO answer must be at the beginning of your response.",
                    "Read the PR diffs, description, and title and answer the question with only YES or NO. Also use the following category background information to assist with your applicability decision. Here is some background about unprefixed commands / grammar misrecognition. Prefixes are needed when defining a large number of short commands in order to ensure clear use. Grammar is the mapping between spoken form and command(s).  For this prompt, consider the following requirements: Are common words used without prefixes (e.g., blue instead of format blue), are there two or more rules in the .talon files that have spoken forms that sound identical or have high overlap, and are there commands of one or two frequently used words without prefix. This category focuses on ensuring that commands and grammar rules are clearly distinguishable and not prone to being misinterpreted by a parser, compiler, or user input system.  Ambiguity can arise when elements are not explicitly marked (e.g. missing prefixes or insufficient structure), leading to incorrect parsing or unintended behavior. This category is \”Avoid unprefixed commands / grammar misrecognition\”. Answer YES if the provided diffs, description, and/or title contains or introduces any of the previously defined unprefixed commands or grammar issues. Otherwise, answer NO. The YES/NO answer must be at the beginning of your response.",
                    "Read the PR diffs, description, and title and answer the question with only YES or NO. Also use the following category background information to assist with your applicability decision. Here is some background information on the .talon file taxonomy. It consists of the following, a header at the top that defines information that is contextually important like what applications this file is active for and what OS is being used, and a body that lists out code commands that will be done when the file is called and the application specified is being used. Abstractions are the building blocks, like scripts, commands, and contexts, that  translate voice commands into specific computer actions. They allow someone to define what a spoken phrase does and when it should work, depending on the app being used. Given that information, analyze the diff and determine if there is unrelated functionality that has been written based on the existing .talon files and Talon abstractions. Additionally, read the stated purpose from the PR title and description and determine if the code being submitted differs in functionality from the title and description. If there is unrelated functionality answer YES, otherwise answer NO. The YES/NO answer must be at the beginning of your response.",
                    "Read the PR diffs, description, and title and answer the question with only YES or NO. Also use the following category background information to assist with your applicability decision. Consider the Talon documentation when making this analysis: Rules can be anchored or unanchored. Talon has a system that detects when a user is and isn't speaking which it uses to break up microphone input into a sequence of 'utterance blocks'. So if you said \"first bit ... other ... bits\" ('...' means a sufficiently long pause), then Talon might turn this into three utterance blocks: [\"first bit\", \"other\", \"bits\"]. Anchoring a rule requires that it occur at the start or end (or both) of an utterance block. For example if the following command were added to the Talon Community user file set ^my command: \"first\" and you said \"my command air bat cap\" then Talon would insert \"firstabc\". \"air bat cap my command\" on the other hand would only produce \"abc\" (and maybe a misrecognition) because 'my command' was not at the start of your utterance. If other command$: \"second\" were defined and you said \"air bat cap other command\" you'd get \"abcsecond\". If you said \"other command air bat cap\" you'd just get \"second\". Because the command matched and had the $ suffix, the rest of your utterance was thrown away. In general you shouldn't anchor rules since it prevents the user from chaining them together (like we were doing with our examples and the air bat cap commands). Aside from special circumstances you really only consider anchoring when you have a command you wouldn't chain (e.g. switching from command to dictation mode), or you really want to prevent the command from being called by accident. This diff analysis revolves around how anchoring is used, and analyzing when it is or is not necessary: Answer YES if the provided diffs, description, and/or title overuses anchoring or includes anchors that could be removed. Otherwise, answer NO. The YES/NO answer must be at the beginning of your response.",
                    "Read the PR diffs, description, and title and answer the question with only YES or NO. Also use the following category background information to assist with your applicability decision. Here is some background information on what proper spoken forms are. Talon provides abstractions for defining flexible command spoken forms. A list maps spoken forms to corresponding values. A capture can use combinations of lists, other captures, and explicit spoken forms to map what the user says to values. As an example, a list can map the names of numbers to their numeric values to allow defining voice commands that take a number as an argument, such as allowing saying \"delete twenty five\" to press the delete key 25 times or \"delete thirty\" to press the delete key 30 times. Captures allow for complex combinations of spoken forms to allow for flexible commands. This category is \”Use proper spoken forms\”. Answer YES if the provided diffs, description, and/or title do not have the previously proper spoken forms defined as lists and/or captures. Otherwise, answer NO. The YES/NO answer must be at the beginning of your response.",
                    "Read the PR diffs, description, and title and answer the question with only YES or NO. Also use the following category background information to assist with your applicability decision. Here is some background about what counts as complex text insertion. Common actions to looks for that will denote a complex text insertion include the following: Code that inserts or auto inserts a literal string longer than a single word (e.g., insert(\“text\”), auto_insert(\"text\")), code that use the key action multiple times after an insert action to simulate physical key presses (e.g., key(space), key(right)), and code containing multiple edit actions to move the cursor(e.g., edit.right()). Answer YES if the provided diffs, description, and/or title contains any of the previously mentioned actions related to complex text insertion. Otherwise, answer NO. The YES/NO answer must be at the beginning of your response.",
                    "Read the PR diffs, description, and title and answer the question with only YES or NO. Familiarize yourself with the .talon files and this information about Talon: The BODY part of a command is implemented in Talonscript, a simple statically typed language. In terms of the Talonscript itself, the syntax can be thought of as a very limited subset of Python. Determine if the submitted code is not using any syntax that makes the overall code easier to read and modify in the future. These can be things like writing components of the code on the same line instead of splitting it into multiple lines, or if a feature could be used to make the code more compact. If any of these TalonScript convenience features could be used and are not answer YES, otherwise answer NO. The YES/NO answer must be at the beginning of your response.",
                    "Read the PR diffs, description, and title and answer the question with only YES or NO. Also use the following category background information to assist with your applicability decision. Consider the Talon documentation when making this analysis: Re-using a voice phrase (the spoken form, i.e. the text before the : in a .talon file) to change behavior is possible, but it's a brittle technique and should be used cautiously. Why it's brittle Exact spoken form match required (including whitespace). To reliably replace an existing command, your override must use the exact same spoken form as upstream, character-for-character, including spaces and punctuation. Any deviation (extra/missing space, different optional tokens, captures, etc.) does not replace the upstream command; it defines a second command. If both variants load, Talon uses undocumented tie-breakers. When two non-identical spoken forms are present, both commands load, and Talon applies internal, undocumented rules to decide which one runs when you speak the phrase. Context headers do not affect this tie-break. Upstream changes can silently break your override. Even if the words you say haven't changed, any tweak to the spoken form you copied (whitespace, punctuation, or how optionals/captures are structured) can stop your override from applying or create competing definitions with unpredictable selection. Note: Context headers control when a file is active; they do not determine priority if the spoken forms differ. To truly replace a command, the spoken form must be identical to the upstream definition to replace it. Example: changing touch to not close the mouse grid Suppose the Talon Community file defines mouse.talon without a context header: touch: mouse_click(0) # Close the mouse grid if open user.grid_close() # End any open drags # Touch automatically ends left drags so this is for right drags specifically user.mouse_drag_end() We can see the user.grid_close() action is called to close the grid after clicking the mouse. Also, note the lines starting with '#' characters are called comments. They are just there for documentation and will not be otherwise processed by Talon. If we wanted to stop the user.grid_close() behaviour we could create a new .talon file and put in the following contents: macOS-only variant. This only replaces upstream if the spoken form \"touch:\" is identical (including whitespace). Otherwise both load and Talon chooses via undocumented rules, ignoring the context header. os: mac touch: mouse_click(0) # End any open drags # Touch automatically ends left drags so this is for right drags specifically user.mouse_drag_end() Notice that we've given it a context header. Because this context header is more specific (i.e. it has more rules in it) this implementation of \"touch\" will take precedence over the original. The implementation just has the user.grid_close() line and associated comment removed. In general, you can use this technique by just making a version of the .talon file you want to override and putting in more redundant rules to make it the more specific version. In addition to \"os: \" some other redundant filters you can add are \"mode: command\" (assuming you want to define the command in the default 'command' mode) and \"speech.engine: wav2letter\" (assuming you're not using Dragon). Important: This replaces the original only if the line touch: is exactly the same as upstream. If upstream later changes that line you may end up with two competing definitions. Safer patterns (recommended) Override actions instead of phrases. Find the action the phrase ultimately calls (see Finding what a command does) and implement your own version in your user files. This decouples you from upstream spoken forms. Add a new phrase instead of replacing the old one. Map your preferred behavior to a new, clearly named spoken form. This avoids collisions entirely. Vendor and disable the original. If you must replace the original phrase, copy the upstream file into your custom fileset, make your edits there, and ensure the upstream copy does not load in your environment. With only one definition, there are no tie-breakers. If you still choose a phrase-level override: Copy the upstream spoken form exactly (including whitespace/punctuation and any captures/optionals). Keep the override in a separate fileset so you can track upstream changes. After updates, diff spoken forms before pulling in changes. Use the REPL/introspection tools to verify which command actually fired. Other Voice Commands per the Contributing.MD page in the community repository: P01 - Prefer [object][verb] rather than [verb][object] for new commands. For example 'file save' is better than 'save file'. It may not sound as natural, but it helps for grouping related commands in lists and avoiding conflicting names. P02 - Use browser.host matcher for web apps. Though this matcher requires a browser extension on some operating systems it is the only unambiguous way of referring to a web app. This diff analysis considers whether the diff is following community standard practices for spoken forms or not: Answer YES if the provided diffs, description, and/or title do not follow community conventions (e.g. noun/verb patterns or capitalization of single letters) and should be updated. Otherwise, answer NO. The YES/NO answer must be at the beginning of your response."]

In [ ]:
cat_1_ex_1 = """
The line "ctx.lists["self.kotlin_modifier"] = kotlin_modifiers" in file lang/kotlin/kotlin.py for PR 1408 received this maintainer comment:
"I think we should probably modernize this to use the code_keywords stuff? See JavaScript for an example https://github.com/talonhub/community/blob/main/lang/tags/keywords.talon" 
that marked it as a PR that belongs under the "Build off of the existing programming language support when implementing support for a new language" category.
"""

cat_1_ex_2 = """
The line "ctx.lists["self.go_types"] = {" in file lang/go/go.py for PR 906 received this maintainer comment: "Please use the standard type list that we use in eg Python and Javascript" 
that marked it as a PR that belongs under the "Build off of the existing programming language support when implementing support for a new language" category.
"""

cat_2_ex_1 = """
An example from a portion of a diff file that was marked by a maintainer as belonging to the "Remove useless/unused code" category is:
-# def file_manager_current_path(): 
-#     title = ui.active_window().title 
-#     if "~" in title: 
-#         title = os.path.expanduser(title) 
-#     if title in directories_to_remap: 
-#         title = directories_to_remap[title] 
-#     if title in directories_to_exclude: 
-#         title = None 
- 
-#     return title 
"""

cat_2_ex_2 = """
An example from a portion of a diff file that was marked by a maintainer as belonging to the "Remove useless/unused code" category is:
-# def file_manager_show_properties(): 
-#     \"""Shows the properties for the file\""" 
-        # actions.auto_insert("// ") 
"""

cat_3_ex_1 = """
PR 1020 had lines 14-23 that were deleted in windows_shell.py (deleting that existing functionality to a more core file could have had cascading effects on other core functions that relied on code in that file,
those effects were not properly documented anywhere) in commit 7a4a11bde9cf8feff84a717c18b8ce1d350751a9 with maintainer comment: "What additions did you have in mind for 'ls'? I presume you wanted to do more than change the current implementation while providing no new features.
Also FYI, as implemented this will break functionality in some of the window's shells. See this implementation for example: https://github.com/knausj85/knausj_talon/blob/main/apps/windows_command_processor/command_processor_win.py#L86" 
that marked it as a PR that belongs under the "Properly handle changes to existing functionality" category.
"""

cat_3_ex_2 = """
PR 2049 had line 32 that was added in lang/java/java.py (adding the String type incorrectly as a boxed type) in commit 7b4907f74f739105e4617cfc00279ce25fc0184e with maintainer comment: 
"String is not a boxed type in Java. It should already be in another list and usable." that marked it as a PR that belongs under the "Properly handle changes to existing functionality" category.
"""

cat_4_ex_1 = """
Before and after examples from a portion of a diff file that was marked by a maintainer as belonging to the "Avoid unnecessarily complex code" category is:
before:
class BrowserActions:
    def address():
        # Split title by space, check each token and token[1: -1] (it might be in brackets) for valid url.
        # Prioritize last one if multiple are valid, return empty string if none is valid.
        tokens = (
            url[1:-1] if not is_url(url) else url
            for url in reversed(actions.win.title().split(" "))
        )
        return next((url for url in tokens if is_url(url)), "")
After:
class BrowserActions:
    def address() -> str:
        title: str = actions.win.title()
        if not title:
            return ""

        # We expect the URL to either be prepended or appended to the page title.
        first, *tokens = title.split()
        if is_url(first):
            return first

        # Prioritize last one if multiple are valid.
        for url in reversed(tokens):
            if is_url(url):
                return url
            # The URL may be in [brackets].
            unbracketed = url[1:-1]
            if is_url(unbracketed):
                return unbracketed

        # None were valid.
        return ""
"""

cat_4_ex_2 = """
PR 906 for lines 24-26: 
" 
+ ctx.lists["self.go_pointers"] = {
+     "pointer": "*",
+ }
"
in the file lang/go/go.py in the commit c6d1c1f6b8c3049a14d8789d8201f8099f4e14fe with maintainer comment: "why do we need a list for this?"
that marked as as belonging to the "Avoid unnecessarily complex code" category. 
"""

cat_5_ex_1 = """
The lines "
+ empty square:
+  insert("[]")
+ empty brace:
+  insert("{}")
" were added in file text/symbols.talon for PR 888 received this maintainer comment "What would you use these two commands for?" that marked it as a PR that belongs under the "Explain unclear decisions" category.
"""

cat_5_ex_2 = """
The lines "
+ def code_state_if():
+  actions.user.insert_between("if ", " ")
" were added in file lang/go/go.py for PR 906 received this maintainer comment "What is the " " doing here?" that marked it as a PR that belongs under the “Explain unclear decisions” category.
"""

cat_6_ex_1 = """
An example from a portion of a diff file that was marked by a maintainer as belonging to the "Provide broader operating systems support" category is:
+os: linux  
+and app.name: Cursor 
"""

cat_6_ex_2 = """
An example from a portion of a diff file that was marked by a maintainer as belonging to the "Provide broader operating systems support" category is:
+@ctx.action_class("browser")  
+class BrowserActions: 
+    def show_extensions(): 

+        actions.app.tab_open() 
+        actions.browser.go("arc://extensions") 
"""

cat_7_ex_1 = """
PR 1451 had lines 4-10 that were deleted and line 4 "apps.obsidian = "app.name: Obsidian"" that was added in obsidian.py in commit bae7fa035bddfc073c02eef471741a073a58eb01 with maintainer comments: "Please just do mod.apps.obsidian" and 
"Do you think creating another context for this is better than calling actions.next? :-) thought we were trying to avoid having to make changes every time you override code.language" 
that marked it as a PR that belongs under the "Override behavior by overriding actions when possible" category.
"""

cat_7_ex_2 = """
PR 679 had line 19 "
type \\{user.code_type\\}:
" that was added in lang/java/java.talon in commit 79012f44b7097387ba43839da087cf4da80429d2 with mainainer comments: 
"This is already defined in lang/tags/generic.talon, why redefine it here? I know the pull request says you want to add a space for consistency, but what is it you're being consistent with?" and 
"It looks like we're using the code_type list for Python and Typescript and hence the generic 'type' command which doesn't insert the space. I'd suggest we be consistent with that and don't override this here so that people get the same behaviour across languages.
What do you think? You could easily override type to add the space in your personal configuration and we can keep things more consistent in knausj." 
that marked it as a PR that belongs under the "Override behavior by overriding actions when possible" category. 
"""

cat_8_ex_1 = """
PR 83 had file misc/1password_global.talon with added line 2
"
#todo: tags
+ app: 1Password.exe
-
password fill: user.password_fill()
password show: user.password_show()
"
with maintainer comment : "I think this breaks the desired functionality: these commands should be available in e.g. safari, Firefox, etc. I might be missing something though? This might be a better use case for tags, but not 100 percent sure."
where the maintainers wanted the added line to be more generalized to other applications to have a password manager functionality that marked it as a PR that belongs under the "Put generalizable functionality behind a tag and provide overrideable actions for it" category.
"""

cat_8_ex_2 = """
PR 12 has lines in code/jetbrains.py
"mod = Module()
@mod.action_class
class Actions:
    def idea(commands: str):
        \"""Send a command to Jetbrains product\"""
        command_list = commands.split(",")
        print("executing jetbrains", commands)
        global extendCommands
        extendCommands = command_list
        for cmd in command_list:
            send_idea_command(cmd)
            time.sleep(0.1)

    def idea_num(command: str, number: str, zero_okay: bool = False):
        \"""Sends a command with numbers to Jetbrains product\"""
        print(number)
        if int(number) == 0 and not zero_okay:
            print("Not sending, arg was 0")
            return

        formatted = command % number
        send_idea_command(formatted)
        global extendCommands
        extendCommands = []
"
with apps/jetbrains.talon lines
"
+ (action | please): user.knausj_talon.code.jetbrains.idea("action GotoAction")
+ (action | please) <dgndictation>:
+  user.knausj_talon.code.jetbrains.idea("action GotoAction")
+  insert(dictate.join_words(dgndictation))
"

with maintainer comment: "jetbrains is an IDE, yes? Wondering if we can define some common actions, similar to like window_management.talon or tabs.talon, and define those per-app"
that marked it as a PR that belongs under the "Put generalizable functionality behind a tag and provide overrideable actions for it" category.
"""

cat_9_ex_1 = """
The lines "
+ # apps to exclude from running list
+ excludes = set()
" were added in file core/app_switcher/app_switcher.py for PR 1385 received this maintainer comment:
"Probably worth adding a comment that any line without a comma is excluded? This is a bit different from several of the other CSVs, where it's assumed the spoken form and result are the same in this case." 
that marked it as a PR that belongs under the "Document complex code" category. 
""" 

cat_9_ex_2 = """
The line "
+ # websites.csv in your user/community/settings directory.
" was added in file core/websites_and_search_engines/websites_and_search_engines.py for PR 1361 received this maintainer comment:
"I think this is actually potentially more confusing in so far as settings/ is relative to the community repo, whereas user/community/ is "outside" it, which means the community repo doesn't really know about what that path will be. 
So for instance, a lot of people end up cloning the community repo with a different name, especially if they fork, and so in those cases users/community/ will be wrong. 
With that in mind I would suggest may be just adding an extra line that gives an example of a possible location for it. So some thing like: # Please do not edit these defaults. 
Instead, add / edit your own entries in # settings/websites.csv in your user directory. For example, if you cloned the # talonhub community repo as "community", then the file would be in <talon folder>/user/community/settings/websites.csv." 
that marked it as a PR that belongs under the "Document complex code" category. 
""" 

cat_10_ex_1 = """
An example from a portion of a diff file that was marked by a maintainer as belonging to the "Prefix short commands that may misrecognize with other commands" category is:
+ctx.lists["self.go_pointers"] = {
+    "pointer": "*",
+}
""" 

cat_10_ex_2 = """
An example from a portion of a diff file that was marked by a maintainer as belonging to the "Prefix short commands that may misrecognize with other commands" category is:
+    "ABBREVIATION": (NOSEP, every_word(lambda w: w[0])),
"""

cat_11_ex_1 = """
PR 1827 had all lines changed across the 6 files and 6 commits. Maintainer comment: "From the community backlog session — apologies for overlooking this PR for several months. 
Could you please split your changes into a single PR for each file you are changing to make it easier to review? Thanks! (We now have a contribution guideline P07 that addresses this — it wasn't there when you submitted this PR…"
that marked it as a PR that belongs under the "Override behavior by overriding actions when possible" category.
""" 

cat_11_ex_2 = """
PR 743 had line 33 in code.py in commit 4a1ac24a41f17d92d4c06bd349e52c00505716ad. This addition to the code should not be in the PR because it has nothing to do with the intended purpose stated in the PR title and description, 
which is what marked it as a PR that belongs under the "Unrelated changes should not go in the same pull request." category.
"""

cat_12_ex_1 = """
Before and after examples from a portion of a diff file that was marked by a maintainer as belonging to the "Only use command anchoring when necessary." category for removing anchoring is:
Before: 
    @ctx.capture("number", rule=f"(<digits> | [<digits>] <user.number_scaled>)$") 
After: 
    @ctx.capture("number", rule=f"(<digits> | [<digits>] <user.number_scaled>)")
""" 

cat_12_ex_2 = """
PR 679 had line 51 added:
" 
+ ^annotate with$: insert("@")
"
in lang/java/java.talon in commit 79012f44b7097387ba43839da087cf4da80429d2 with maintainer commentd: "This doesn't need to be anchored, it probably would work better unanchored." and
"The '^' prefix means the command has to be at the start of an utterance. I.e. '^my command: "hello "' means I can say 'my command word world' and get 'hello world' written out, but I can't say 'word world my command' (I'll get 'world' out, and then maybe a misrecognition or nothing).
The '$' suffix means the rest of the utterance after the command is discarded. So if I had 'my command$: "hello "' and said "my command word world ten dot" I'd just get 'hello ' out.
Generally you try not to use anchors since it prevents people from fluidly making utterances as long as they want. For example when editing code I'm often able to queue up a bunch of edits to one line then move to a different line and start editing there in one utterance. The main limiting factor is the potential misrecognitions from Talon requiring backtracking." 
that marked it as a PR that belongs under the "Only use command anchoring when necessary." category.
""" 

cat_13_ex_1 = """
The line "
+ "estadd": "estadd",
" in file lang/stata/stata.py for PR 1401 received this maintainer comment:
"Just want to confirm you pronounce these literally "estadd" and not "E S T add". If the latter, should change the wording to spell it out." 
that marked it as a PR that belongs under the "Use proper spoken forms" category. 
""" 

cat_13_ex_2 = """
The line "
+ 'you 64': 'uint64',
" in file lang/proto/proto.py for PR 765 received this maintainer comment:
"Numbers in spoken forms don't work, you'll need to spell them out eg "you sixty four"." 
that marked it as a PR that belongs under the "Use proper spoken forms" category. 
"""    

cat_14_ex_1 = """
An example from a portion of a diff file that was marked by a maintainer as belonging to the "Use snippets for complex text insertion" category is:
+    def code_state_for(): 
+        actions.insert("forval  {\n\n}\n") 
+        actions.key("up:2 tab up right:3")
""" 

cat_14_ex_2 = """
An example from a portion of a diff file that was marked by a maintainer as belonging to the "Use snippets for complex text insertion" category is:
+ empty dubstring: user.insert_between('"', '"') 
+ empty escaped (dubstring|dub quotes): user.insert_between('\"', '\"') 
+ empty string: user.insert_between("'", "'") 
+ empty escaped string: user.insert_between("\'", "\'")
""" 

cat_15_ex_1 = """
PR 743 had a maintainer gave this as an example: 
"delete key(left left left left)
   use key(left:4)" 
that marked it as a PR that belongs under the "Take advantage of TalonScript convenience features" category.
"""

cat_15_ex_2 = """
PR 743 had lines 57-59: 
"
has type: insert(": ") 
ref: insert("&")
ref mute: insert("&mut")
" in lang/rust/rust.talon in commit fa6e15900189979f09fb115593d4a3f67dcb715b had maintainer comments:
"No need to call insert() for literal strings. You can do something like: ref mute: "&mut"" and 
"It's probably not necessary to change unless you feel like it, just good to know about the short syntax. Stylistically we also only use the short syntax if its the only thing in the Talonscript body (otherwise it's a bit harder to read)."
that marked it as a PR that belongs under the "Take advantage of TalonScript convenience features" category.
""" 

cat_16_ex_1 = """
Before and after examples from a portion of a diff in file lang/stata/stata.py:
Before:
    "v c e cluster": "vce(cluster)",
After: 
    "V C E cluster": "vce(cluster)",
that was marked by a maintainer with the comment:
"I could be wrong, but I think usually for single letters we usually use capitals, although I'm not sure it actually matters. Can just ignore this suggestion if it works ok as is."
which marked it as belonging to the "Follow community standard practices for spoken forms" category.
"""

cat_16_ex_2 = """
Before and after examples from a portion of a diff in file lang/stata/stata.py:
Before:
    "v c e robust": "vce(robust)",
After: 
    "V C E robust": "vce(robust)",
that was marked by a maintainer with the comment:
"I could be wrong, but I think usually for single letters we usually use capitals, although I'm not sure it actually matters. Can just ignore this suggestion if it works ok as is."
which marked it as belonging to the "Follow community standard practices for spoken forms" category.
"""

In [ ]:
#Diff explanation sourced from the diff Wikipedia: https://en.wikipedia.org/wiki/Diff#Unified_format
diff_def = """
The format starts with the same two-line header as the context format, except that the original file is preceded by "---" and the new file is preceded by "+++". Following this are one or more change hunks that contain the line differences in the file. The unchanged, contextual lines are preceded by a space character, addition lines are preceded by a plus sign, and deletion lines are preceded by a minus sign.

A hunk begins with range information and is immediately followed with the line additions, line deletions, and any number of the contextual lines. The range information is surrounded by double at signs, and combines onto a single line what appears on two lines in the context format (above). The format of the range information line is as follows:

@@ -l,s +l,s @@ optional section heading

The hunk range information contains two hunk ranges. The range for the hunk of the original file is preceded by a minus symbol, and the range for the new file is preceded by a plus symbol. Each hunk range is of the format l,s where l is the starting line number and s is the number of lines the change hunk applies to for each respective file. In many versions of GNU diff, each range can omit the comma and trailing value s, in which case s defaults to 1. Note that the only really interesting value is the l line number of the first range; all the other values can be computed from the diff.

The hunk range for the original should be the sum of all contextual and deletion (including changed) hunk lines. The hunk range for the new file should be a sum of all contextual and addition (including changed) hunk lines. If hunk size information does not correspond with the number of lines in the hunk, then the diff could be considered invalid and be rejected.

Optionally, the hunk range can be followed by the heading of the section or function that the hunk is part of. This is mainly useful to make the diff easier to read. When creating a diff with GNU diff, the heading is identified by regular expression matching.[13]

If a line is modified, it is represented as a deletion and addition. Since the hunks of the original and new file appear in the same hunk, such changes would appear adjacent to one another.[14] An occurrence of this in the example below is:

-check this dokument. On
+check this document. On

To successfully separate the file names from the timestamps, the delimiter between them is a tab character. This is invisible on screen and can be lost when diffs are copy/pasted from console/terminal screens. 
"""

In [ ]:
file_def = """ 

#File type explanation sourced from the community member reference document here: https://github.com/FireChickenProductivity/TalonVoiceReferenceDocuments/blob/main/development_big_picture.md, the talon framework overview from the talon wiki here: https://talon.wiki/Customization/Talon%20Framework/talon-framework-overview/, and the .talon file documentation from the talon wiki here: https://talon.wiki/Customization/talon-files  

.talon vs .py explanation: 

Talon will automatically try to load everything inside a user folder when it starts up. Any folders or file names seen in Talon user file sets (e.g. Talon Community) were chosen by the authors of that package. Talon also monitors files in the user directory, and will automatically reload them if they're changed by printing a log message. This reloading is convenient when working on scripts/configuration as you generally don't have to restart Talon for it to pick up changes. 

There are two kinds of configuration/scripting files (.py and .talon), because originally all Talon configuration/scripting was done using Python, but over time it was decided that the addition of a framework specific file type would be beneficial. To a first approximation .talon files provide a succinct way of mapping spoken commands to behavior. .py files on the other hand provide the implementation of behavior and other functionality used by .talon files. 

  

.talon file explanation: 

.talon files consist of a context header and body separated by a line consisting entirely of a "-". A context header determines the circumstances in which the .talon file is active. The header may be omitted to have the file globally active in command mode. The body may define voice commands, define key bindings, set settings, or activate tags. 

 Key .talon syntax includes: 

Context header (e.g., mentioned and shown above), command mapping that maps a spoken phrase to an action (e.g., phrase: action()), variables use curly braces to insert captures that can be lists or numbers (e.g., {user.list}), comments are denoted by a hash symbol (e.g., # This is a comment), the body must be indented, variable names in the body are derived directly from the names inside the < > or { } in the rule, a voice command has the format RULE: BODY, where RULE determines what words activate the command, and BODY defines what the command does when activated: 

# -------- RULE ----------------- ------ BODY ------- 
([channel] unread next | goneck): key(alt-shift-down)  

This command, for example, will press the shortcut alt-shift-down whenever you say either “channel unread next”, “unread next”, or “goneck”. 

Rules have a versatile syntax that is like a word based regex: 

| Syntax | Description | Matches |
| :--- | :--- | :--- |
| foo | Words | “foo” | 
| [foo] | Optional | “foo” or null (nothing) | 
| foo* | Zero or more | “”, “foo”, “foo foo” |
| foo+ | One or more | “foo”, “foo foo” | 
| foo \\| bar | Choice | “foo”, “bar” | 
| (foo) | Precedence/grouping | “foo” |
| {some_list} | List | Depends on the list |
| <some_capture> | Capture | Depends on the capture |
| ^foo | Start anchor | See below | 
| foo$ | End anchor | See below |

Rules can be anchored or unanchored. Talon has a system that detects when a user is and isn't speaking which it uses to break up microphone input into a sequence of 'utterance blocks'. So if you said "first bit ... other ... bits" ('...' means a sufficiently long pause), then Talon might turn this into three utterance blocks: ["first bit", "other", "bits"]. Anchoring a rule requires that it occur at the start or end (or both) of an utterance block. 

Be sure to understand that for .talon file diffs, a line starting with - (minus then space) is a deleted line of code, whereas a single - appearing alone in the original file structure is the context header and body separator. 

 

.py file explanation 

Talon Python files can work with Talon's abstractions by importing from talon. Module objects allow defining things like actions and settings while you can use Context objects to override stuff defined by a Module in specific contexts, such as overriding an action when a specific application is focused or on a specific operating system. 

Talon provides abstractions for defining flexible command spoken forms. A list maps spoken forms to corresponding values. A capture can use combinations of lists, other captures, and explicit spoken forms to map what the user says to values. As an example, a list can map the names of numbers to their numeric values to allow defining voice commands that take a number as an argument, such as allowing saying "delete twenty five" to press the delete key 25 times or "delete thirty" to press the delete key 30 times. Captures allow for complex combinations of spoken forms to allow for flexible commands. 

Talon provides constructs for understanding context. A Tag is either active or not. Tags can be activated and deactivated through commands or as a consequence of other context information. A Scope can have arbitrary values. This can allow matching on properties of a scope or specific scope values. A Mode defines a set of commands users can chain together. Users typically define a mode if they only want commands from that mode active in specific contexts, such as an exam mode limiting the available commands to a small, approved subset. 

Context is important. A lot of functionality needs to be overridden based on the nature of the current operating system or application. Additionally, having voice commands available in contexts where they are not relevant makes command misrecognitions more common. 

Key .py syntax includes: 

Context is defined through context objects (e.g., ctx = Context(), ctx.matches = ), variables like lists are defined using square brackets (e.g., list = [“list_item”]), command mapping is handled through defining the action and setting it to a context header (e.g. ctx.action = (app: name)), comments are denoted by a hash symbol (e.g., # This is a comment), actions are typically defined within a class decorated with @mod.action_class, context overrides use the ctx.action_class decorator. 

""" 

In [ ]:
def sys_stats():
    # CPU Usage
    cpu_percent = round(psutil.cpu_percent(interval=0.1), 2)
    # Sys Ram Usage
    ram_percent = round(psutil.virtual_memory().percent, 2)
    # GPU Usage
    gpu_percent = 0
    if curr_id == -1:
        gpu_percent = 0
    try:
        command = [
                "nvidia-smi", 
                f"--id={curr_id}", 
                "--query-gpu=memory.used,memory.total", 
                "--format=csv,nounits,noheader"
        ]
        raw_output = subprocess.check_output(command)
        decode_output = raw_output.decode('utf-8').strip()
        used, total = decode_output.split(', ')
        gpu_percent = round((int(used) / int(total)) * 100, 2)
        
    except:
        gpu_percent = -1
    
    return cpu_percent, ram_percent, gpu_percent

In [ ]:
# Run the Qwen3-4B model on the given PR diffs for each category prompt
# This function was built off of the Qwen3 Quickstart guide on how to run the model generate content based on inputs found here: https://huggingface.co/Qwen/Qwen3-8B#quickstart 
def run_model(diff_def, diffs, description, title, category_prompts, shotNum, thinkMode):
    results = {}
    cat_count = 1
    token_count = 0
    total_time = 0
    pattern = r"Decision:\s*(YES|NO)"
    examples = {
        "1": [cat_1_ex_1, cat_1_ex_2],
        "2": [cat_2_ex_1, cat_2_ex_2],
        "3": [cat_3_ex_1, cat_3_ex_2],
        "4": [cat_4_ex_1, cat_4_ex_2],
        "5": [cat_5_ex_1, cat_5_ex_2],
        "6": [cat_6_ex_1, cat_6_ex_2],
        "7": [cat_7_ex_1, cat_7_ex_2],
        "8": [cat_8_ex_1, cat_8_ex_2],
        "9": [cat_9_ex_1, cat_9_ex_2],
        "10": [cat_10_ex_1, cat_10_ex_2],
        "11": [cat_11_ex_1, cat_11_ex_2],
        "12": [cat_12_ex_1, cat_12_ex_2],
        "13": [cat_13_ex_1, cat_13_ex_2],
        "14": [cat_14_ex_1, cat_14_ex_2],
        "15": [cat_15_ex_1, cat_15_ex_2],
        "16": [cat_16_ex_1, cat_16_ex_2]
    }

    #Logging messages
    if shotNum == 0:
        print("Zero-shot prompting running")
    elif shotNum == 1:
        print("One-shot prompting running")
    elif shotNum == 2:
        print("Two-shot prompting running")
    
    if thinkMode.strip().lower() == "yes":
        print("Thinking mode enabled") 
    elif thinkMode.strip().lower() == "no":
        print("Thinking mode disabled")

    for category in category_prompts:
        # Get category examples
        cat_exs = examples.get(str(cat_count), [])

        # Handle zero-, one-, and two-shot prompts
        if shotNum == 0:
            # Prepare the model input for zero-shot
            messages = [{"role": "user", "content": f"You must format your final decision as: Decision: YES or Decision: NO. Analyze the diff definition here: {diff_def} to understand how to read these PR Diffs: {diffs}. PR title: {title}. PR description: {description}. Use this prompt: {category}"}]
        elif shotNum == 1:
            # Prepare the model input for one-shot
            messages = [{"role": "user", "content": f"You must format your final decision as: Decision: YES or Decision: NO. Analyze the diff definition here: {diff_def} to understand how to read these PR Diffs: {diffs}. PR title: {title}. PR description: {description}. Utilize this example of a previous PR and its diffs, code, filename, and/or maintiner comment that falls under this category: {cat_exs[0]}. Use this prompt: {category}"}]
        else:
            # Perform the model input for two-shot
            messages = [{"role": "user", "content": f"You must format your final decision as: Decision: YES or Decision: NO. Analyze the diff definition here: {diff_def} to understand how to read these PR Diffs: {diffs}. PR title: {title}. PR description: {description}. Utilize these examples of previous PRs and their diffs, code, filename, and/or maintiner comments that fall under this category: {cat_exs[0]} and {cat_exs[1]}. Use this prompt: {category}"}]
        
        # Handle thinking mode on or off
        if thinkMode.strip().lower() == "yes":
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=True # Enable thinking
            )

            # Change max_tokens depending on thinkMode
            maxTokens = 1024 # Allow thinking and room for YES/NO output

        else:
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False # Disable thinking
            )

            maxTokens = 10 # Enforce YES/NO output


        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

        # Throughput measurement sourced from: https://discuss.pytorch.org/t/best-way-to-measure-timing/39496 
        # Synchronize before starting timer if GPU is in use
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start_time = perf_counter()
        
        # conduct the category verification
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=maxTokens
        )

        # Synchronize before ending timer if GPU is in use
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        end_time = perf_counter()


        #Calculate total time from start and end of model generation
        model_time = end_time - start_time
        total_time += model_time
        
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
        # Count number of tokens generated
        token_count += len(output_ids)
        
        content = tokenizer.decode(output_ids, skip_special_tokens=True).strip()
        # Filter out only Decision: YES/NO
        match = re.findall(pattern, content, re.IGNORECASE)

        # Get the last Decision: YES/NO output from the regex match 
        if match:
            output = match[-1].upper()
        else:
            # Get the last YES/NO in output if DECISION: YES/NO
            backup_pattern = r"\b(YES|NO)\b"
            backup_match = re.findall(backup_pattern, content, re.IGNORECASE)
            if backup_match:
                output = backup_match[-1].upper()
            else:
                output = "NONE"

        # Keep track of categories through 1-16 incremented
        results[str(cat_count)] = output
        print(f"Processed Category {cat_count}")
        cat_count+=1

        # Clean up GPU 
        torch.cuda.empty_cache()
        
    return results, total_time, token_count

In [ ]:
# Create a dictionary of the human-determined YES/NO categories based on the values in the true_cats list of lists to add to the csv
def get_true_cats(curr_true_cats, total_cats):
    truths = {}
    for i in range(1, total_cats + 1):
        if i in curr_true_cats:
            truths[str(i)] = "YES"
        else:
            truths[str(i)] = "NO"
    return truths        

In [ ]:
def write_cat_csv(final_results, filename, total_categories):
    headers = ["PR"]
    for i in range(1, total_categories + 1):
        headers += [f"Result Category {i}"]
    for i in range(1, total_categories + 1):
        headers += [f"Truth Category {i}"]
    
    # Write to csv
    with open(filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(headers)
    
        #Get the prnum, model results and truth results per PR
        for result in final_results:
            pr = result.get("PR")
            results = result.get("Results", {})
            truths = result.get("Truths", {})
    
            # Add row with PR number
            row = [pr]
            # Add results per catefory into their own columns
            for i in range(1, total_categories + 1):
                row.append(results.get(str(i), "N/A"))
            #Add truths per category into their own columns
            for i in range(1, total_categories + 1):
                row.append(truths.get(str(i), "N/A"))
    
            writer.writerow(row)

    print(f"Successfully saved results to {filename}")

In [ ]:
def write_time_csv(final_results, filename):
    headers = ["PR", "Total Model Time", "Total Tokens"]
    
    # Write to csv
    with open(filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(headers)
    
        #Get the prnum, model results and truth results per PR
        for result in final_results:
            pr = result.get("PR")
            time = result.get("Total Model Time", 0)
            tokens = result.get("Total Tokens", 0)
    
            # Add row with PR number
            row = [pr, time, tokens]   
            writer.writerow(row)

    print(f"Successfully saved results to {filename}")

In [ ]:
def write_sys_csv(final_results, filename):

    headers = ["PR", "GPU Name", "CPU %", "Mem %", "GPU %"]
    
    # Write to csv
    with open(filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(headers)
    
        #Get the prnum, model results and truth results per PR
        for result in final_results:
            pr = result.get("PR")
            name = result.get("GPU Name", "N/A")
            cpu = result.get("CPU", "N/A")
            mem = result.get("MEM", "N/A")
            gpu = result.get("GPU", "N/A")
    
            # Add row with PR number
            row = [pr, name, cpu, mem, gpu]   
            writer.writerow(row)

    print(f"Successfully saved results to {filename}")

In [ ]:
# Set list of PRs and category_prompts for each dataset
train_set_commits = {"1451": "d2288b6e059fe33fb38b15ec08f3c5fa93e78f49", "1408": "ba6bb5f09d17442e10f2ace6d4d1a113272b84d6", "1385": "fdf321ed18f5efd6a7cb4380921358adc6a2b225", "743": "fa6e15900189979f09fb115593d4a3f67dcb715b"}
train_true_cats = [[6,7], [4], [11,4,7,8], [6], [6,3], [6,3], [3,12], [8], [7,2,4], [3,4], [8], [6], [1], [6], [10], [5], [14,5,3,1,10,8,2,4,9], [3], [5,4], [2], [3], [3], [5], [15], [13,16,2,1,5,14], [12,7,10,13], [9], [12,13,4], [15,14,4,1,10,2,11], [2], [4], [9], [11], [3], [2], [2], [6], [4], [2], [11]]
train_PRs = [1200, 844, 545, 107, 337, 803, 92, 83, 1451, 1020, 12, 5, 1408, 1326, 243, 888, 906, 1075, 1102, 1846, 1357, 2049, 1581, 142, 1401, 679, 1385, 765, 743, 1068, 1752, 1361, 1827, 1833, 2013, 1768, 1549, 636, 641, 1328]

val_set_commits = {"576": "e8b29c2d342e5b0ba2f561f1271ab8678b57bdca", "752": "30ca4e85443f3424d99d9778aa9558e092b4b63a", "1578": "efb49b64a6f1e1a2ae1c06d80c7e7fb91b5bbc4c"}
val_true_cats = [[8], [8], [8], [11], [2,16], [7], [6], [1,2], [2,3], [2], [2], [3], [11], [10,1], [1,2], [12], [2], [1,3,4,10,11,14], [4], [6], [3,5,11], [6,2], [5], [2,3,7], [2], [7,15], [3], [8], [4,5,6,9], [4], [5,16], [6], [2], [3,4,14], [2], [13], [5,13], [15], [9], [3,10]]
val_PRs = [157, 279, 328, 329, 444, 493, 527, 576, 588, 623, 632, 638, 645, 647, 735, 752, 758, 786, 798, 808, 956, 985, 1050, 1273, 1344, 1364, 1431, 1455, 1578, 1627, 1638, 1697, 1763, 1820, 1890, 2127, 578, 97, 1182, 928]

test_set_commits = {"246": "b5c86a6536b6492c565b636a73e72620ed7c2a89", "470": "4734f22cec72746fb81d21bed11acb7cb895d4d6", "863": "24732235cd78547afcff796d5c4e883f27e514c6", "1885": "369f1ef6f3457c9782f27f14436defb5f62dc12e", "796": "f05b367db833b62fbbceaef082f257cd1b388647", "799": "5bff25f5ba03c4cc63b65e1b27956c79ed06dba3"}
test_true_cats = [[6,7,8], [8], [8], [5], [5,6], [3], [3], [11], [15,16], [3], [9], [2,5], [3], [8,12], [3,4,5,9], [2,4,5,6,12], [1,4,14], [12], [3,9,15], [3,14], [2,4], [2,4], [16], [5], [2,3,4], [10], [4], [5], [3], [9], [5], [4,5], [1], [1,3], [3,10,13], [13], [7], [11], [11], [11], [11]]
test_PRs = [46, 102, 242, 246, 296, 386, 420, 470, 547, 555, 624, 648, 656, 680, 696, 771, 777, 794, 802, 830, 863, 879, 1089, 1219, 1313, 1333, 1375, 1641, 1677, 1813, 1850, 1885, 968, 304, 796, 799, 1143, 810, 574, 1436, 1457]

In [ ]:
def model_output(commits, true_cats, dataset_PRs, cat_filename, time_filename, sys_filename, shots, thinking):
    cat_results = []
    time_results = []
    sys_results = []
    total_categories=len(category_prompts)

    try: 
        for i, pr in enumerate(dataset_PRs):
            # Get the system stats to log
            cpu, ram, gpu = sys_stats()
            sys_results.append({"PR": pr, "GPU Name": gpu_name, "CPU": cpu, "MEM": ram, "GPU": gpu
            })

            print(f"Processing PR {pr}")
            # Call the get_diffs function to grab the diffs per PR 
            diffs, description, title = get_diffs(pr, commits)
            
            # Call the run_model function to have the model determine category applicability for zero-shot and no thinking
            output, pr_time, pr_tokens = run_model(diff_def, diffs, description, title, category_prompts, shots, thinking) 
        
            #Get the true categories from the true_Cats list of lists
            if i < len(true_cats):
                curr_true_cats = true_cats[i]
            # Call the function to get the true category per PR
            truth = get_true_cats(curr_true_cats, total_categories)
            
            # Append output to the cat_results and time_results per PR
            cat_results.append({"PR": pr, "Results": output, "Truths": truth})
            time_results.append({"PR": pr, "Total Model Time": pr_time, "Total Tokens": pr_tokens})

    # Handle if an OOM or other error occurs
    except Exception as e: 
        print(f"Error: {e}")
    
    finally:
    #Handle writing to csv regardless of error
        if cat_results:
            write_cat_csv(cat_results, cat_filename, total_categories)
        if time_results:
            write_time_csv(time_results, time_filename)
        if sys_results:
            write_sys_csv(sys_results, sys_filename)
        else:
            print("No results were generated to save.")

In [ ]:
#Train RQ1 Dataset for Zero-shot with Thinking
model_output(train_set_commits, train_true_cats, train_PRs, "category_model_results_train_zero_think.csv", "time_model_results_train_zero_think.csv", "sys_model_results_train_zero_think.csv", 0, "yes")

In [ ]:
#Validate RQ1 Dataset for Zero-shot with Thinking
model_output(val_set_commits, val_true_cats, val_PRs, "category_model_results_validate_zero_think.csv", "time_model_results_validate_zero_think.csv", 0, "yes")

In [ ]:
#Train RQ2 Dataset for Few-shot with Thinking
model_output(train_set_commits, train_true_cats, train_PRs, "category_model_results_train_few_think.csv", "time_model_results_train_few_think.csv", 2, "yes")

In [ ]:
#Validate RQ2 Dataset for Few-shot with Thinking
model_output(val_set_commits, val_true_cats, val_PRs, "category_model_results_validate_few_think.csv", "time_model_results_validate_few_think.csv", 2, "yes")

In [ ]:
#Test RQ1 Dataset for Zero-shot with Thinking
model_output(test_set_commits, test_true_cats, test_PRs, "category_model_results_test_zero_think.csv", "time_model_results_test_zero_think.csv", "sys_model_results_test_zero_think.csv", 0, "yes")

In [ ]:
#Test RQ2 Dataset for Few-shot with Thinking
model_output(test_set_commits, test_true_cats, test_PRs, "category_model_results_test_few_think.csv", "time_model_results_test_few_think.csv", "sys_model_results_test_few_think.csv", 2, "yes")